In [ ]:
devtools::load_all()  # ensure package + C++ are loaded

set.seed(123)

## --- 1) Simulate data with known truth ---------------------------------------
n  <- 1000
p  <- 3
X  <- cbind(1, rnorm(n), rnorm(n))           # include intercept
beta0  <- c(0.7, -1.2, 0.8)
sigma0 <- 1.3

mu  <- as.numeric(X %*% beta0)
y   <- mu + rnorm(n, 0, sigma0)

## helper: true conditional tau-quantile under Normal noise
q_true <- function(tau) mu + qnorm(tau, 0, sigma0)

taus <- c(0.1, 0.5, 0.9)

## --- 2) Fit exAL static MCMC for each tau ------------------------------------
## previous run saved (3000 - 1500)/2 = 750 draws; mirror that here:
##   n.burn = 1500, n.mcmc = 750, thin = 2
fit_one <- function(tau) {
  exal_static_mcmc(
    y, X, p0 = tau,
    b0 = rep(0, ncol(X)), V0 = diag(1e4, ncol(X)),  # weak Normal prior
    a_sigma = 0.0001, b_sigma = 0.0001,             # same ultra-weak as before
    n.burn = 1500, n.mcmc = 750, thin = 2,
    verbose = TRUE
  )
}

fits <- lapply(taus, fit_one)
names(fits) <- paste0("tau=", taus)

## quick chain sanity (dimensions)
stopifnot(all(sapply(fits, function(f) inherits(f$samp.beta, "mcmc"))))
stopifnot(all(sapply(fits, function(f) ncol(as.matrix(f$samp.beta)) == ncol(X))))

## --- 3) Posterior summaries for beta/sigma/gamma -----------------------------
summ_one <- function(fit) {
  B <- as.matrix(fit$samp.beta)      # nd x p
  S <- as.numeric(fit$samp.sigma)    # nd
  G <- as.numeric(fit$samp.gamma)    # nd
  list(
    beta_med  = apply(B, 2, median),
    beta_ci   = t(apply(B, 2, quantile, c(0.05, 0.95))),
    sigma_med = median(S),
    gamma_med = median(G)
  )
}
sums <- lapply(fits, summ_one)

cat("\n=== Parameter recovery (posterior medians) ===\n")
for (k in seq_along(taus)) {
  cat(sprintf("\n-- %s --\n", names(fits)[k]))
  print(rbind(truth = beta0, post_median = sums[[k]]$beta_med))
  cat(sprintf("sigma0=%.3f | sigma_med=%.3f\n", sigma0, sums[[k]]$sigma_med))
  cat(sprintf("gamma_med=%.3f (with Normal data, should shrink near 0)\n", sums[[k]]$gamma_med))
}

## --- 4) Posterior predictive tau-quantile vs true tau-quantile --------------
## For each i and each kept draw d:
##   s ~ N^+(0,1), v ~ Exp(rate = 1/sigma_d), Z ~ N(0,1)
##   y_rep = x_i' beta_d + λ(gamma_d) σ_d s + A(gamma_d) v + sqrt(B(gamma_d) σ_d v) * Z
pp_tau_quantile <- function(fit, X, tau, K = 40) {
  B      <- as.matrix(fit$samp.beta)   # nd x p
  sigmad <- as.numeric(fit$samp.sigma) # nd
  gammad <- as.numeric(fit$samp.gamma) # nd
  nd     <- nrow(B)
  n      <- nrow(X)
  p      <- ncol(X)
  stopifnot(length(sigmad) == nd, length(gammad) == nd, ncol(B) == p)

  # Precompute A, B, lambda per draw for this tau
  A_d   <- vapply(gammad, function(g) A.fn(tau, g), numeric(1))
  BB_d  <- vapply(gammad, function(g) B.fn(tau, g), numeric(1))
  C_d   <- vapply(gammad, function(g) C.fn(tau, g), numeric(1))
  lam_d <- C_d * abs(gammad)

  # (n x nd) matrix of x_i' beta_d
  M <- X %*% t(B)

  qs <- numeric(n)
  for (i in 1:n) {
    m_d <- M[i, ]  # length nd

    # Draw s_i,d,k, v_i,d,k, Z_i,d,k  as nd x K matrices (fresh per i)
    s_mat <- matrix(abs(rnorm(nd * K)), nrow = nd, ncol = K)
    v_vec <- rexp(nd * K, rate = 1 / rep(sigmad, each = K))
    v_mat <- matrix(v_vec, nrow = nd, ncol = K, byrow = TRUE)
    Z_mat <- matrix(rnorm(nd * K), nrow = nd, ncol = K)

    # μ = m_d + λ_d σ_d s + A_d v
    mu_mat <- matrix(m_d, nrow = nd, ncol = K) +
              sweep(s_mat, 1, lam_d * sigmad, "*") +
              sweep(v_mat, 1, A_d, "*")

    # sd = sqrt(B_d σ_d v)
    sd_mat <- sqrt(sweep(v_mat, 1, BB_d * sigmad, "*"))

    yrep_mat <- mu_mat + sd_mat * Z_mat
    qs[i] <- as.numeric(quantile(yrep_mat, probs = tau, names = FALSE))
  }
  qs
}

pp_qs <- lapply(seq_along(taus), function(k) pp_tau_quantile(fits[[k]], X, taus[k], K = 40))
names(pp_qs) <- names(fits)

## --- 5) Diagnostics: RMSE vs true tau-quantile -------------------------------
rmse <- function(a, b) sqrt(mean((a - b)^2))
rmse_tau <- sapply(seq_along(taus), function(k) rmse(pp_qs[[k]], q_true(taus[k])))

cat("\n=== Predictive tau-quantile RMSE (exAL vs truth under Normal data) ===\n")
print(setNames(round(rmse_tau, 4), names(fits)))

In [ ]:

## --- 6) Quick visual for one tau (windowed) ---------------------------------
k_show <- which(taus == 0.1)  # choose a tau to visualize
qq_est <- pp_qs[[k_show]]
qq_tru <- q_true(taus[k_show])

win  <- 150
i2   <- n
i1   <- max(1L, i2 - win + 1L)
xx   <- i1:i2

col_true <- "#1F2937"
col_est  <- "#0EA5E9"

op <- par(no.readonly = TRUE); par(mar = c(3.2, 4.2, 2.4, 1.2))
plot(xx, qq_tru[xx], type = "l", col = col_true, lwd = 2,
     main = sprintf("Predictive %s vs. True %s quantile (window)", names(fits)[k_show], names(fits)[k_show]),
     xlab = "index", ylab = expression(q[tau](x)))
lines(xx, qq_est[xx], col = col_est, lwd = 2)
legend("topleft", c("true quantile", "exAL posterior predictive quantile"),
       col = c(col_true, col_est), lwd = 2, bty = "n")
par(op)


In [ ]:

## --- 6) Quick visual for one tau (windowed) ---------------------------------
k_show <- which(taus == 0.5)  # choose a tau to visualize
qq_est <- pp_qs[[k_show]]
qq_tru <- q_true(taus[k_show])

win  <- 150
i2   <- n
i1   <- max(1L, i2 - win + 1L)
xx   <- i1:i2

col_true <- "#1F2937"
col_est  <- "#0EA5E9"

op <- par(no.readonly = TRUE); par(mar = c(3.2, 4.2, 2.4, 1.2))
plot(xx, qq_tru[xx], type = "l", col = col_true, lwd = 2,
     main = sprintf("Predictive %s vs. True %s quantile (window)", names(fits)[k_show], names(fits)[k_show]),
     xlab = "index", ylab = expression(q[tau](x)))
lines(xx, qq_est[xx], col = col_est, lwd = 2)
legend("topleft", c("true quantile", "exAL posterior predictive quantile"),
       col = c(col_true, col_est), lwd = 2, bty = "n")
par(op)


In [ ]:

## --- 6) Quick visual for one tau (windowed) ---------------------------------
k_show <- which(taus == 0.9)  # choose a tau to visualize
qq_est <- pp_qs[[k_show]]
qq_tru <- q_true(taus[k_show])

win  <- 150
i2   <- n
i1   <- max(1L, i2 - win + 1L)
xx   <- i1:i2

col_true <- "#1F2937"
col_est  <- "#0EA5E9"

op <- par(no.readonly = TRUE); par(mar = c(3.2, 4.2, 2.4, 1.2))
plot(xx, qq_tru[xx], type = "l", col = col_true, lwd = 2,
     main = sprintf("Predictive %s vs. True %s quantile (window)", names(fits)[k_show], names(fits)[k_show]),
     xlab = "index", ylab = expression(q[tau](x)))
lines(xx, qq_est[xx], col = col_est, lwd = 2)
legend("topleft", c("true quantile", "exAL posterior predictive quantile"),
       col = c(col_true, col_est), lwd = 2, bty = "n")
par(op)


In [ ]:
devtools::load_all()  # ensure package + C++ are loaded

set.seed(123)

## --- 1) Simulate data with known truth ---------------------------------------
n  <- 1000
p  <- 3
X  <- cbind(1, rnorm(n), rnorm(n))           # include intercept
beta0  <- c(0.7, -1.2, 0.8)
sigma0 <- 1.3

mu  <- as.numeric(X %*% beta0)
y   <- mu + rnorm(n, 0, sigma0)

## helper: true conditional tau-quantile under Normal noise
q_true <- function(tau) mu + qnorm(tau, 0, sigma0)

taus <- c(0.1, 0.5, 0.9)

## --- 2) Fit exAL static VB (LDVB) for each tau --------------------------------
fit_one <- function(tau) {
  exal_static_LDVB(
    y, X, p0 = tau,
    b0 = rep(0, ncol(X)), V0 = diag(1e4, ncol(X)),  # weak Normal prior
    a_sigma = 0.0001, b_sigma = 0.0001,             # same ultra-weak as MCMC tests
    max_iter = 2000, tol = 1e-5,
    n_samp_xi = 300,
    verbose = TRUE
  )
}

fits <- lapply(taus, fit_one)
names(fits) <- paste0("tau=", taus)

## --- 2.0) ELBO traces: inspect & visualize -----------------------------------
elbo_traces <- lapply(fits, function(f) {
  tr <- f$misc$elbo
  if (is.null(tr)) numeric(0) else as.numeric(tr)
})
names(elbo_traces) <- names(fits)

# Print quick ELBO diagnostics per tau
cat("\n=== ELBO diagnostics ===\n")
for (nm in names(elbo_traces)) {
  tr <- elbo_traces[[nm]]
  it <- fits[[nm]]$iter
  conv <- fits[[nm]]$converged
  if (length(tr) == 0) {
    cat(sprintf("%s : (no ELBO trace found)\n", nm))
  } else {
    del <- diff(tr)
    rel_last <- if (length(tr) >= 2) (tr[length(tr)] - tr[length(tr)-1]) / (1e-8 + abs(tr[length(tr)-1])) else NA_real_
    cat(sprintf(
      "%s : iters=%d | converged=%s | ELBO_final=%.6f | ΔELBO_last=%.3e | min(ΔELBO)=%.3e\n",
      nm, it, conv, tail(tr, 1), ifelse(is.finite(rel_last), rel_last, NA_real_), ifelse(length(del), min(del), NA_real_)
    ))
  }
}

# ELBO plotting helper
plot_elbo <- function(k) {
  nm <- names(elbo_traces)[k]
  tr <- elbo_traces[[k]]
  if (!length(tr)) {
    warning(sprintf("No ELBO trace for %s", nm)); return(invisible(NULL))
  }
  plot(seq_along(tr), tr, type = "l", lwd = 2,
       xlab = "CAVI iteration", ylab = "ELBO",
       main = sprintf("ELBO evolution — %s", nm))
  grid()
}

# Example: visualize ELBO for all taus
op <- par(no.readonly = TRUE)
par(mfrow = c(1, length(taus)), mar = c(3.2, 4.2, 2.4, 1.2))
invisible(lapply(seq_along(taus), plot_elbo))
par(op)


## --- 2.1) Small helper: sample draws from VB posterior ------------------------
vb_posterior_draws <- function(fit, nd = 1000) {
  # q(beta) ~ N(m, V)
  m  <- as.numeric(fit$qbeta$m)
  V  <- as.matrix(fit$qbeta$V)
  p  <- length(m)
  # robust chol
  Uc <- tryCatch(chol(V), error = function(e) NULL)
  if (is.null(Uc)) Uc <- chol(V + 1e-10 * diag(p))
  Zb <- matrix(rnorm(nd * p), nd, p)
  B  <- sweep(Zb %*% t(Uc), 2, m, `+`)  # nd x p

  # q(eta, ell) ~ N([eta_hat, ell_hat], Sigma) ; gamma = L + (U-L) * logistic(eta), sigma = exp(ell)
  mu2   <- c(fit$qsiggam$eta_hat, fit$qsiggam$ell_hat)
  Sig2  <- as.matrix(fit$qsiggam$Sigma)
  U2    <- tryCatch(chol(Sig2), error = function(e) NULL)
  if (is.null(U2)) U2 <- chol(Sig2 + 1e-8 * diag(2))
  Z2    <- matrix(rnorm(nd * 2), nd, 2)
  pars  <- sweep(Z2 %*% t(U2), 2, mu2, `+`)
  eta   <- pars[, 1]
  ell   <- pars[, 2]
  Lbnd  <- as.numeric(fit$misc$bounds[1])
  Ubnd  <- as.numeric(fit$misc$bounds[2])
  gamma <- Lbnd + (Ubnd - Lbnd) * plogis(eta)
  sigma <- exp(ell)

  list(beta = B, sigma = sigma, gamma = gamma)
}

## --- 3) Posterior summaries for beta/sigma/gamma via VB draws -----------------
summ_one <- function(fit, nd = 2000) {
  dr <- vb_posterior_draws(fit, nd = nd)
  B  <- dr$beta
  S  <- dr$sigma
  G  <- dr$gamma
  list(
    beta_med  = apply(B, 2, median),
    beta_ci   = t(apply(B, 2, quantile, c(0.05, 0.95))),
    sigma_med = median(S),
    gamma_med = median(G)
  )
}
sums <- lapply(fits, summ_one)

cat("\n=== Parameter recovery (VB posterior medians) ===\n")
for (k in seq_along(taus)) {
  cat(sprintf("\n-- %s --\n", names(fits)[k]))
  print(rbind(truth = beta0, post_median = sums[[k]]$beta_med))
  cat(sprintf("sigma0=%.3f | sigma_med=%.3f\n", sigma0, sums[[k]]$sigma_med))
  cat(sprintf("gamma_med=%.3f (with Normal data, should shrink near 0)\n", sums[[k]]$gamma_med))
}

## --- 4) Posterior predictive tau-quantile vs true tau-quantile ---------------
## Use posterior draws from VB to approximate predictive distribution
pp_tau_quantile_from_draws <- function(B, sigmad, gammad, X, tau, K = 40) {
  nd <- nrow(B); n <- nrow(X); p <- ncol(X)
  stopifnot(length(sigmad) == nd, length(gammad) == nd, ncol(B) == p)

  # Precompute A, B, lambda per draw for this tau
  A_d   <- vapply(gammad, function(g) A.fn(tau, g), numeric(1))
  BB_d  <- vapply(gammad, function(g) B.fn(tau, g), numeric(1))
  C_d   <- vapply(gammad, function(g) C.fn(tau, g), numeric(1))
  lam_d <- C_d * abs(gammad)

  # (n x nd) matrix of x_i' beta_d
  M <- X %*% t(B)

  qs <- numeric(n)
  for (i in 1:n) {
    m_d <- M[i, ]  # length nd

    # Draw s_i,d,k, v_i,d,k, Z_i,d,k  as nd x K matrices (fresh per i)
    s_mat <- matrix(abs(rnorm(nd * K)), nrow = nd, ncol = K)
    v_vec <- rexp(nd * K, rate = 1 / rep(sigmad, each = K))
    v_mat <- matrix(v_vec, nrow = nd, ncol = K, byrow = TRUE)
    Z_mat <- matrix(rnorm(nd * K), nrow = nd, ncol = K)

    # μ = m_d + λ_d σ_d s + A_d v
    mu_mat <- matrix(m_d, nrow = nd, ncol = K) +
              sweep(s_mat, 1, lam_d * sigmad, "*") +
              sweep(v_mat, 1, A_d, "*")

    # sd = sqrt(B_d σ_d v)
    sd_mat <- sqrt(sweep(v_mat, 1, BB_d * sigmad, "*"))

    yrep_mat <- mu_mat + sd_mat * Z_mat
    qs[i] <- as.numeric(quantile(yrep_mat, probs = tau, names = FALSE))
  }
  qs
}

pp_qs <- lapply(
  seq_along(taus),
  function(k) {
    dr <- vb_posterior_draws(fits[[k]], nd = 1500)  # draws for predictive approx
    pp_tau_quantile_from_draws(dr$beta, dr$sigma, dr$gamma, X, taus[k], K = 40)
  }
)
names(pp_qs) <- names(fits)

## --- 5) Diagnostics: RMSE vs true tau-quantile -------------------------------
rmse <- function(a, b) sqrt(mean((a - b)^2))
rmse_tau <- sapply(seq_along(taus), function(k) rmse(pp_qs[[k]], q_true(taus[k])))

cat("\n=== Predictive tau-quantile RMSE (VB exAL vs truth under Normal data) ===\n")
print(setNames(round(rmse_tau, 4), names(fits)))

## --- 6) Quick visuals (windowed) ---------------------------------------------
plot_window <- function(k_show, win = 150) {
  qq_est <- pp_qs[[k_show]]
  qq_tru <- q_true(taus[k_show])
  i2 <- n; i1 <- max(1L, i2 - win + 1L); xx <- i1:i2
  col_true <- "#1F2937"; col_est <- "#0EA5E9"
  op <- par(no.readonly = TRUE); par(mar = c(3.2, 4.2, 2.4, 1.2))
  plot(xx, qq_tru[xx], type = "l", col = col_true, lwd = 2,
       main = sprintf("Predictive %s vs. True %s quantile (window)", names(fits)[k_show], names(fits)[k_show]),
       xlab = "index", ylab = expression(q[tau](x)))
  lines(xx, qq_est[xx], col = col_est, lwd = 2)
  legend("topleft", c("true quantile", "exAL VB predictive quantile"),
         col = c(col_true, col_est), lwd = 2, bty = "n")
  par(op)
}

# Show for tau = 0.1, 0.5, 0.9
plot_window(which(taus == 0.1))
plot_window(which(taus == 0.5))
plot_window(which(taus == 0.9))
